# Data Mining Lecture 6
## Association rules and co-occurrence

Open in Google Colab and save a copy. Run the setup cell, then work through the sections in order. Pause at each discussion question, make a prediction before each live demo, and use Shift+Enter to run code. The examples use illustrative baskets and the Iris dataset; the separate lab uses Wine. No installation or data upload is needed.

**Outcomes:** Calculate and interpret support, confidence and lift; generate simple rules; evaluate limitations and practical usefulness.

**Suggested approach:** Keep the transaction unit and denominator visible throughout. A rule is a descriptive pattern in observed data—not a causal claim, guarantee, or recommendation to act.

In [ ]:
# Purpose: import the tools used throughout the notebook and verify the runtime is ready.
import pandas as pd  # Build and inspect transaction tables.
import numpy as np  # Available for numerical work in optional extensions.
from itertools import combinations  # Visit each unordered pair of items once.
from sklearn.datasets import load_iris  # Load a small built-in dataset; no upload is needed.
from IPython.display import display  # Render tables clearly in Jupyter/Colab.
print("Setup ready")

## A01 From clusters to associations

Clustering groups similar observations. Association-rule mining looks for items or attributes that occur together across transactions. Both are descriptive methods: they summarize observed data, and neither establishes causation. The choice of transaction unit matters because it determines what ‘together’ means.

**Discuss in pairs:** Think of a setting you know (shopping, health services, media, education, or web use). What could count as one transaction? Which two events might co-occur? Who could use that information, and what decision—if any—might it inform? What other explanation could account for the co-occurrence?

## A02 Learning outcomes

By the end, you should be able to define transactions and itemsets; calculate support, confidence and lift; compare directional rules; identify misleading rules; and use pandas to investigate and communicate a rule.

**Discuss:** Rate your confidence for each outcome from 1 (new) to 5 (can explain and apply it). Which is new, and which builds on earlier summaries such as proportions or conditional probabilities? What would you need to see or practise to move one rating up by one point?

## A03 Opening diagnostic

Consider three claims: (1) a 90% confidence rule always proves a useful association; (2) a rule implies causation; and (3) A → B means the same thing as B → A. Each statement requires examination.

**Discuss before continuing:** For each claim, decide ‘always true’, ‘sometimes true’, or ‘false’. Give a reason or a counterexample. In particular, what does confidence use as its denominator? What would you compare it with to know whether 90% is unusually high? Revisit your answers after A10.

## A04 Transactions and items

A transaction is the unit counted once, such as a purchase receipt. An item is a named event or attribute in that unit. A transaction is represented by the distinct items present; repeated quantities are not counted twice in a binary basket. The transaction unit is part of the analysis design, not a neutral implementation detail.

**Discuss:** What changes if the unit is a receipt rather than a customer over a year? Would ‘tea and biscuits’ then mean products bought together in one visit, or products ever bought by the same person? How might repeat customers, basket size, time window, or missing records affect counts? State a transaction definition for your opening example.

## A05 Itemsets and rules

An itemset is a set of one or more items, such as {tea, biscuit}. A rule A → B compares an antecedent A with a consequent B. The arrow represents a conditional association; transaction data alone do not show that A happened before B. With multi-item rules, A and B may each contain more than one item and are normally disjoint.

**Discuss:** In {tea} → {biscuit}, identify antecedent and consequent. Translate the rule into an accurate sentence beginning ‘Among transactions containing tea…’. Why would ‘tea causes biscuit purchases’ be an unjustified translation? What additional design or evidence might be needed for a causal question?

## A06 A small worked basket dataset

Five illustrative transactions contain tea, coffee and biscuits. T1={tea,biscuit}; T2={tea,biscuit}; T3={tea}; T4={coffee,biscuit}; T5={coffee}. Each transaction is one basket. Treat the basket records as binary: presence or absence, not quantity.

**Discuss and predict:** How many baskets contain tea, biscuits and both tea and biscuits? Write your counts before running the cell. Which quantity will be the denominator for support? What information is lost by representing each basket as binary?

**Live demonstration:** Predict the result, then run the next cell with Shift+Enter. Check your counts against the displayed matrix.

In [ ]:
# Purpose: encode the five example baskets as a binary transaction matrix.
# Each row is one transaction; each column is an item; True means present.
baskets = [set(['tea','biscuit']), set(['tea','biscuit']), set(['tea']), set(['coffee','biscuit']), set(['coffee'])]
items = sorted(set.union(*baskets))  # Find all distinct item names for matrix columns.
toy = pd.DataFrame([{item: item in basket for item in items} for basket in baskets], index=[f'T{i}' for i in range(1,6)])
display(toy.astype(int))  # Show 1/0 instead of True/False to make counts easy to read.
# Purpose: compare the marginal item counts with their joint co-occurrence count.
print('Total:',len(toy),'tea:',toy['tea'].sum(),'biscuit:',toy['biscuit'].sum(),'both:',(toy['tea'] & toy['biscuit']).sum())

## A07 Support measures coverage

Support(A → B) means support(A ∪ B): the fraction of all transactions containing the antecedent and consequent together. For tea → biscuit, support is 2/5 = 0.40, or 40%. The denominator is all valid transactions, not only transactions containing tea.

**Discuss:** What does 40% support tell a store manager—and what does it not tell them? If the store has 1,000 baskets, what joint count corresponds to this support? How could a change in the transaction window or the definition of a basket change the result?

## A08 Confidence is conditional

Confidence(A → B) = count(A and B) / count(A). It describes the proportion of A transactions that also contain B. For tea → biscuit, confidence is 2/3 ≈ 66.7%. It is an empirical conditional proportion, not a probability that one product causes another.

**Discuss:** Explain confidence without using a formula, using the five baskets. Now imagine confidence is 100% because the only tea basket also contains biscuits. Is that strong evidence for a reliable pattern? What additional count or baseline would you want?

## A09 Direction matters

Confidence(B → A) uses count(B) as its denominator. In this example tea and biscuit both occur in three baskets, so both directions happen to have confidence 2/3. Equal confidence is not a general property.

**Discuss and calculate:** If A appears in 4 baskets, B in 2, and both in 2, find confidence(A → B) and confidence(B → A). Which rule has the larger confidence and why? Does that mean its joint co-occurrence is larger? Distinguish the shared numerator from the direction-specific denominator.

## A10 Lift compares with the baseline

Lift(A → B) = confidence(A → B) / support(B), provided the denominator is nonzero. For tea → biscuit, (2/3)/(3/5) = 10/9 ≈ 1.11. Lift equals 1 under empirical independence, exceeds 1 for positive association and is below 1 for negative association. Pairwise lift is symmetric even though confidence is directional.

**Discuss:** Why can high confidence still have lift near 1? Suppose 95% of all transactions contain B and 95% of A transactions contain B. Is A especially informative about B? Explain lift relative to a baseline. What does lift say about association, and what can it not say about practical value or causality?

**Live demonstration:** Predict the result, then run the next cell with Shift+Enter. Identify each denominator in the printed values.

In [ ]:
# Purpose: calculate support, confidence, the consequent's baseline prevalence, and lift for tea → biscuit.
n = len(toy)  # Total transactions: the support denominator.
a = toy['tea']  # Boolean indicator for transactions containing the antecedent.
b = toy['biscuit']  # Boolean indicator for transactions containing the consequent.
joint = (a & b).sum()  # Number containing both items.
support = joint/n  # Joint coverage across all transactions.
confidence = joint/a.sum()  # Among tea baskets, fraction also containing biscuits.
baseline = b.mean()  # Fraction of all baskets containing biscuits.
lift = confidence/baseline  # Compare conditional prevalence with its baseline.
print(f'joint={joint}, total={n}; support={support:.3f}; confidence={confidence:.3f}; baseline={baseline:.3f}; lift={lift:.3f}')

## A11 Calculate and cross-check

The same lift can be calculated as support(A and B)/(support(A) × support(B)). It is symmetric for a pair, while confidence is directional. Zero antecedent counts make confidence undefined; zero consequent support makes lift undefined.

**Discuss:** Does reversing a rule change its lift? Verify by using the pairwise formula. Why can confidence change while lift does not? A robust calculation should make undefined cases explicit rather than silently divide by zero—what check would you add before calculating the measures?

## A12 Frequent itemsets and thresholds

An itemset is frequent when its observed support meets a specified minimum support. Every subset of a frequent itemset has at least as much support as the itemset. Thresholds manage the search but can remove rare and important patterns. ‘Frequent’ is a chosen filter, not a synonym for ‘interesting’ or ‘important’.

**Discuss:** If a pair has 20% support, can one of its individual items have 10% support? Explain using set containment. Who chooses the minimum-support threshold, and how might business goals, privacy, class imbalance, or rare-but-serious events affect that choice?

## A13 From itemsets to candidate rules

A frequent pair {A,B} gives two directed rules A → B and B → A. A minimum confidence can filter candidates, but it should not replace a baseline comparison or domain judgement. More items lead to many more candidates, increasing computation and the chance of chance findings.

**Discuss:** Why might a high-confidence rule still be unhelpful? Consider a common consequent, a tiny joint count, redundant information, cost of action, or a rule that cannot change a decision. Which measures and contextual information would you require before shortlisting a rule?

## A14 Counts, small samples and spurious findings

A high lift based on one or two cases can be unstable. Searching many rules increases the chance of finding apparently striking patterns. Report transaction counts, thresholds and a follow-up validation plan. If possible, test selected patterns on later or held-out data and consider whether the transaction-generating process changed.

**Discuss:** Would you act immediately on lift 5 from only two joint transactions? What uncertainty remains? Propose a cautious next step. If 10,000 rules were searched, why is the most impressive rule not automatically trustworthy? What should be documented so another analyst can assess selection and reproducibility?

## A15 From measurements to transactions

The live demonstration uses the 150 real Iris measurements. Each flower is one transaction; four original measurements are converted into high/low indicators using dataset-wide medians. Species labels are excluded from the item baskets. Binning simplifies demonstration but discards measurement detail, and the resulting rules depend on the chosen cutoffs.

**Discuss:** What does the meaning of a rule depend on when numeric measurements are turned into categories? What information is lost at a median split? Why exclude species labels for this demonstration, and what could go wrong if an outcome label were mixed into exploratory rules?

**Live demonstration:** Predict the output, then run the next cell. Check the units and explain why these medians are calculated from the measurement columns only.

In [ ]:
# Purpose: load the built-in Iris features, calculate dataset-wide median cutoffs, and inspect them.
iris = load_iris(as_frame=True)  # Built-in data keeps the notebook self-contained.
iris_X = iris.data.copy()  # Use only the four numeric measurements; do not include species labels.
cutoffs = iris_X.median()  # One median per feature defines the high/low teaching bins.
print('Each row is one flower. Species is excluded.')
display(cutoffs.rename('Dataset-wide median (cm)').to_frame())  # Show the thresholds that define every later item.
display(iris_X.head())  # Inspect raw values before converting them to binary items.

## A16 One hot transaction matrix

Each row is a flower and each column is an item. A value of 1 means the item is present, and 0 means it is absent. Each measurement has one of two mutually exclusive high/low items under the stated threshold. Values equal to the median are assigned to HIGH.

**Discuss:** Why is a high-petal-length AND low-petal-length pair impossible for the same flower? What happens to a value exactly equal to a cutoff? If the cutoffs were calculated on a different population or chosen by domain experts, how might the resulting rules change?

**Live demonstration:** Predict the number of rows and columns, and whether the first flower is high or low for each measure, then run the cell.

In [ ]:
# Purpose: convert each numeric measurement into complementary binary HIGH and LOW items.
iris_baskets = pd.DataFrame(index=iris_X.index)  # Preserve one row per flower/transaction.
for col in iris_X.columns:
    # The >= choice assigns values equal to the median to HIGH; < assigns the rest to LOW.
    iris_baskets[col + ' HIGH'] = iris_X[col] >= cutoffs[col]
    iris_baskets[col + ' LOW'] = iris_X[col] < cutoffs[col]
display(iris_baskets.head().astype(int))  # Inspect the one-hot transaction representation.
print('Transactions:',len(iris_baskets),'items:',len(iris_baskets.columns))  # Confirm matrix dimensions.

## A17 Discover rules with pandas

Count each item and pair across the transaction matrix. Calculate both directions for each pair; filter by a declared minimum joint count and lift. The demonstration uses straightforward pandas operations and needs no additional rule-mining package. This intentionally explores pairwise rules only; it is not a complete or optimized frequent-itemset algorithm.

**Discuss before running:** There are eight binary items. How many unordered pairs are considered? Why require a minimum joint count as well as lift greater than 1? What might be missed by the filter, and why should you avoid treating the top-ranked result as a confirmed discovery?

**Live demonstration:** Predict the kind of output, then run the next cell. Inspect the function, locate the joint count and the two different denominators, and note the filters applied after rule generation.

In [ ]:
# Purpose: enumerate pairwise rules from a binary transaction matrix and calculate their measures.
def pair_rules(matrix, min_count=1):
    rows=[]  # Collect one result record per directed rule.
    n=len(matrix)  # All rules use the same total transaction count.
    for a,b in combinations(matrix.columns,2):  # Each unordered pair is visited once.
        ca=int(matrix[a].sum())  # Transactions containing item a.
        cb=int(matrix[b].sum())  # Transactions containing item b.
        joint=int((matrix[a] & matrix[b]).sum())  # Transactions containing both items.
        if joint < min_count or not ca or not cb:  # Skip too-rare pairs and undefined denominators.
            continue
        # Emit both directions: confidence uses the left/antecedent count.
        for left,right,cl,cr in [(a,b,ca,cb),(b,a,cb,ca)]:
            rows.append({'antecedent':left,'consequent':right,'antecedent_count':cl,'consequent_count':cr,'joint_count':joint,'n':n,'support':joint/n,'confidence':joint/cl,'baseline':cr/n,'lift':(joint/n)/((cl/n)*(cr/n))})
    return pd.DataFrame(rows)  # A tabular result makes sorting and interpretation convenient.

# Purpose: require at least 10 joint flowers, keep positive-lift rules, and rank the output.
iris_rules=pair_rules(iris_baskets,min_count=10)
iris_rules=iris_rules[iris_rules.lift>1].sort_values(['lift','joint_count'],ascending=[False,False]).reset_index(drop=True)
display(iris_rules.head(10).round(3))  # Show a short, rounded table for classroom interpretation.

## A18 Interpret a rule in context

Read the actual output: joint count and total, antecedent count, support, confidence, consequent baseline and lift. Report a complete finding in context, together with the median cutoffs. A good interpretation states the population and transaction unit, does not imply causation, and includes counts so readers can judge the amount of evidence.

**Discuss:** What extra information makes a numerical rule understandable to another analyst? Use the displayed first rule to complete: ‘Among ___ flowers, ___ had both ___ and ___. Of flowers with ___, ___% also had ___. The consequent occurs in ___% overall, so the lift is ___.’ What caveat should follow this statement?

**Live demonstration:** Run the cell and compare its output with the rule table. If there are no qualifying rules, revisit the filters and explain what that means rather than assuming a code error.

In [ ]:
# Purpose: print one selected rule with its counts and all measures needed for an interpretation.
if iris_rules.empty:  # Guard against filters that leave no candidate rules.
    print('No rules met the current minimum-count and lift filters; consider what this says about the settings.')
else:
    r=iris_rules.iloc[0]  # Select the highest-ranked rule for a worked interpretation.
    print(f"{r['antecedent']} → {r['consequent']}")
    print(f"Joint {r['joint_count']} / {r['n']}; antecedent {r['antecedent_count']}; consequent {r['consequent_count']}")
    print(f"support {r['support']:.3f}; confidence {r['confidence']:.3f}; baseline {r['baseline']:.3f}; lift {r['lift']:.3f}")

## A19 Ethics and practical usefulness

Association can reflect selection, collection practices or protected characteristics. Observed patterns may be sensitive, unfair, commercially inappropriate or expensive to act upon. A co-occurrence rule alone does not justify profiling or intervention. Consider consent, data minimization, access controls, potential harms, subgroup performance, and whether the proposed use is proportionate.

**Discuss:** What would you check before using a rule involving a sensitive personal attribute? Who might benefit and who might be harmed? Could a proxy attribute reproduce a sensitive pattern even if the sensitive field is removed? Who should review the intended use, and what evidence or governance would you require before deployment?

## A20 Laboratory briefing

The separate lab uses the real scikit-learn Wine dataset. Each wine sample becomes a transaction based on four chemical measurements; median-based binary items are explored with pandas. Wine class is excluded. Students calculate, compare, critique and report their own selected rules.

**Discuss and plan:** Which numerical values from the Iris example must be recalculated for Wine? Before analysing, define the transaction unit, choose and justify binning, set minimum-count/support criteria, and decide how you will validate a selected pattern. How will your report distinguish exploratory evidence from a reliable or useful finding?

## A21 Exit check and sources

Before the lab, distinguish the denominators: support uses all transactions; confidence uses antecedent transactions; lift compares confidence with the consequent baseline. Sources include scikit-learn dataset documentation, pandas and introductory association-rule literature.

**Exit check (work independently, then compare):** A occurs in 8 of 20 baskets, B in 10, and both in 4. Find support, confidence and lift for A → B. Show your denominators, state whether the association is above or below the baseline, and write one sentence that interprets the finding without implying causation. What does the information not tell you?

### Sources
- Iris and Wine dataset documentation: https://scikit-learn.org/stable/datasets/toy_dataset.html
- pandas documentation: https://pandas.pydata.org/docs/
- Agrawal, Imieliński and Swami (1993), *Mining association rules between sets of items in large databases*, https://doi.org/10.1145/170035.170072

The toy basket records are invented for teaching. Iris and Wine are real historical datasets; derived bins and association rules are teaching representations. The Iris results are exploratory illustrations, not validated scientific findings.